In [4]:
from astropy.io import fits
hdulist = fits.open('horseHead.fits')
hdulist.info()
hdr = hdulist[0].header
for key in hdr[:10]:
    print(f"{key:18s}= {hdr[key]}")


Filename: horseHead.fits
No.    Name      Ver    Type      Cards   Dimensions   Format
  0  PRIMARY       1 PrimaryHDU     161   (891, 893)   int16   
  1  er.mask       1 TableHDU        25   1600R x 4C   [F6.2, F6.2, F6.2, F6.2]   
SIMPLE            = True
BITPIX            = 16
NAXIS             = 2
NAXIS1            = 891
NAXIS2            = 893
EXTEND            = True
DATE              = 2014-01-09
ORIGIN            = STScI/MAST
SURVEY            = SERC-ER
REGION            = ER768


In [5]:
#Question 6.2
from astropy.io import fits

def inspect_wcs(filename):
    hdul = fits.open(filename)
    header = hdul[0].header

    try:
        ctype1 = header['CTYPE1']
        ctype2 = header['CTYPE2']
    except KeyError:
        ctype1 = 'UNKNOWN'
        ctype2 = 'UNKNOWN'

    try:
        crval1 = header['CRVAL1']
        crval2 = header['CRVAL2']
    except KeyError:
        crval1 = 0.0
        crval2 = 0.0

    try:
        crpix1 = header['CRPIX1']
        crpix2 = header['CRPIX2']
    except KeyError:
        crpix1 = 0.0
        crpix2 = 0.0

    try:
        cdelt1 = header['CDELT1']
        cdelt2 = header['CDELT2']
    except KeyError:
        try:
            cdelt1 = header['CD1_1']
            cdelt2 = header['CD2_2']
        except KeyError:
            cdelt1 = 1.0
            cdelt2 = 1.0

    print(f"Sky System (CTYPE): {ctype1}, {ctype2}")
    print(f"Projection Center (CRVAL): {crval1}, {crval2}")
    print(f"Reference Pixel (CRPIX): {crpix1}, {crpix2}")
    print(f"Pixel Separation (CDELT/CD): {cdelt1}, {cdelt2}")

    hdul.close()

inspect_wcs('m101.fits')

Sky System (CTYPE): RA---TAN, DEC--TAN
Projection Center (CRVAL): 210.80254413455, 54.347395233845
Reference Pixel (CRPIX): 265.0, 265.0
Pixel Separation (CDELT/CD): -0.00047250309643118, 0.00047262462224515


In [6]:
from astropy.io import fits
import numpy as np
import matplotlib.pyplot as plt

hdul = fits.open('m101.fits')
data = hdul[0].data
bg_slice = data[510:530, 5:25]

In [ ]:
#6.3
import numpy as np
from astropy.io import fits
import matplotlib.pyplot as plt

hdul = fits.open('m101.fits')
data = hdul[0].data

noise_slice = data[100:120, 100:120]

rms = np.sqrt(np.mean(noise_slice**2))
mean = np.mean(noise_slice)
std_dev = np.std(noise_slice)

print(f"RMS: {rms}")
print(f"Mean: {mean}")
print(f"Standard Deviation: {std_dev}")

hdul.close()

In [2]:
#Question 6.5
import numpy as np
from astropy.io import fits

data0 = fits.getdata('noisemap0.fits')
data1 = fits.getdata('noisemap1.fits')
data2 = fits.getdata('noisemap2.fits')

std0 = np.std(data0)
std1 = np.std(data1)
std2 = np.std(data2)

stacked_images = np.stack((data0, data1, data2), axis=0)

mean_image = np.mean(stacked_images, axis=0)

std_mean_image = np.std(mean_image)

stds = np.array([std0, std1, std2])
N = len(stds)
theoretical_std = (1 / N) * np.sqrt(np.sum(stds**2))

print(f"Standard Deviation of Mean Image: {std_mean_image}")
print(f"Theoretical Standard Deviation: {theoretical_std}")
print(f"Diff is: {abs(std_mean_image - theoretical_std)}")

Standard Deviation of Mean Image: 5.773191380233297
Theoretical Standard Deviation: 5.770294583366139
Diff is: 0.002896796867157825


In [ ]:
#Question 6.6
import numpy as np
from astropy.io import fits
from astropy.table import Table

hdul = fits.open('APOGEE_stars.fits')
t = Table(hdul[1].data)

condition = (
    (t['STARFLAG'] == 0) &
    (t['ASPCAPFLAG'] == 0) &
    (t['SNR'] > 10) &
    ((t['GAIAEDR3_PARALLAX'] / t['GAIAEDR3_PARALLAX_ERROR']) > 5) &
    (~np.isnan(t['GAIAEDR3_PHOT_G_MEAN_MAG'])) &
    (~np.isnan(t['GAIAEDR3_PHOT_BP_MEAN_MAG'])) &
    (~np.isnan(t['GAIAEDR3_PHOT_RP_MEAN_MAG']))
)

t['DATAFLAG'] = np.where(condition, 1, 0)

M_G = np.full(len(t), -1.0)
valid = t['DATAFLAG'] == 1
m_G_valid = t['GAIAEDR3_PHOT_G_MEAN_MAG'][valid]
parallax_valid = t['GAIAEDR3_PARALLAX'][valid]

M_G[valid] = m_G_valid + 5 * np.log10(parallax_valid / 1000) + 5
t['ABSOLUTE_MAGNITUDE'] = M_G

t.write('APOGEE_update.fits', overwrite=True)
fits.info('APOGEE_update.fits')
print("\nColumn Names:")
print(updated_table.colnames)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

df = pd.read_csv('gaiaTestData.csv')

max_flux = df['phot_g_mean_flux'].max()
print(max_flux)
print(df[df['phot_g_mean_flux'] == max_flux])

df_sorted = df.sort_values(by='phot_g_mean_flux')

fig, ax = plt.subplots()
ax.scatter(df_sorted['phot_g_mean_flux'], df_sorted['phot_g_mean_mag'])
plt.show()

m0_values = df_sorted['phot_g_mean_mag'] + 2.5 * np.log10(df_sorted['phot_g_mean_flux'])
m0_avg = m0_values.mean()
print(m0_avg)